In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Phase 3 — Model Implementation

In [4]:
import os, gc, time, pickle
import numpy as np
import pandas as pd

OUT_DIR   = "/content/drive/MyDrive/AI_Cheema_Gang/processed"
GRAPH_DIR = os.path.join(OUT_DIR, "graphs")
V2        = os.path.join(OUT_DIR, "cicids2017_v2.parquet")
RESULTS   = os.path.join(OUT_DIR, "results")
os.makedirs(RESULTS, exist_ok=True)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.width", 120)
print("paths set")

paths set


In [5]:
df = pd.read_parquet(V2)
feature_cols = pd.read_csv(os.path.join(OUT_DIR, "feature_columns_v2.csv"))["feature"].tolist()
label_map = pd.read_csv(os.path.join(OUT_DIR, "label_mapping.csv"), index_col=0)["code"].to_dict()
code_to_name = {v: k for k, v in label_map.items()}

print(f"rows     {len(df):,}")
print(f"features {len(feature_cols)}")
print(f"classes  {len(label_map)}")
print(f"split    {df['split'].value_counts().to_dict()}")

rows     2,814,457
features 70
classes  15
split    {'train': 1970113, 'test': 844344}


Sanity check

In [6]:
assert df[feature_cols].isna().sum().sum() == 0, "nulls in features"
assert not np.isinf(df[feature_cols].values).any(), "infinities in features"

tab = pd.crosstab(df["Label"], df["split"])[["train", "test"]]
assert (tab["train"] > 0).all() and (tab["test"] > 0).all(), "a class is missing from a split"

print("feature scale: mean %.3f, std %.3f" % (
    df.loc[df.split=="train", feature_cols].values.mean(),
    df.loc[df.split=="train", feature_cols].values.std()))
print("\nall checks passed\n")
print(tab.to_string())

feature scale: mean 0.012, std 0.977

all checks passed

split                         train    test
Label                                      
BENIGN                      1585842  679647
Bot                            1369     587
DDoS                          89617   38408
DoS GoldenEye                  7205    3088
DoS Hulk                     155794   66769
DoS Slowhttptest               3849    1650
DoS slowloris                  4038    1731
FTP-Patator                    5554    2381
Heartbleed                        7       4
Infiltration                     25      11
PortScan                     111162   47642
SSH-Patator                    4127    1770
Web Attack - Brute Force       1054     453
Web Attack - Sql Injection       14       7
Web Attack - XSS                456     196


#RF with validation set

In [9]:
df["split3"] = df["split"]

for lab, g in df[df["split"] == "train"].groupby("Label", sort=False):
    g = g.sort_values("Timestamp")
    k = max(1, int(len(g) * 0.15))
    df.loc[g.index[-k:], "split3"] = "val"

print(df["split3"].value_counts().to_dict(), "\n")

tab = pd.crosstab(df["Label"], df["split3"])[["train", "val", "test"]]
print(tab.to_string())
print("\nany class missing from a set:",
      tab[(tab == 0).any(axis=1)].index.tolist() or "none")

{'train': 1674601, 'test': 844344, 'val': 295512} 

split3                        train     val    test
Label                                              
BENIGN                      1347966  237876  679647
Bot                            1164     205     587
DDoS                          76175   13442   38408
DoS GoldenEye                  6125    1080    3088
DoS Hulk                     132425   23369   66769
DoS Slowhttptest               3272     577    1650
DoS slowloris                  3433     605    1731
FTP-Patator                    4721     833    2381
Heartbleed                        6       1       4
Infiltration                     22       3      11
PortScan                      94488   16674   47642
SSH-Patator                    3508     619    1770
Web Attack - Brute Force        896     158     453
Web Attack - Sql Injection       12       2       7
Web Attack - XSS                388      68     196

any class missing from a set: none


In [10]:
tr = df[df["split3"] == "train"]
va = df[df["split3"] == "val"]
te = df[df["split3"] == "test"]

X_train = tr[feature_cols].values.astype("float32")
X_val   = va[feature_cols].values.astype("float32")
X_test  = te[feature_cols].values.astype("float32")

y_train_bin, y_val_bin, y_test_bin = (tr["label_binary"].values,
                                      va["label_binary"].values,
                                      te["label_binary"].values)
y_train_mul, y_val_mul, y_test_mul = (tr["label_multi"].values,
                                      va["label_multi"].values,
                                      te["label_multi"].values)

print(f"train {X_train.shape}   val {X_val.shape}   test {X_test.shape}")
print(f"attack rate — train {y_train_bin.mean()*100:.1f}%  "
      f"val {y_val_bin.mean()*100:.1f}%  test {y_test_bin.mean()*100:.1f}%")

train (1674601, 70)   val (295512, 70)   test (844344, 70)
attack rate — train 19.5%  val 19.5%  test 19.5%


In [12]:
from sklearn.ensemble import RandomForestClassifier
from tqdm.auto import tqdm
import joblib

N_TREES, CHUNK = 100, 10

rf_bin = RandomForestClassifier(
    n_estimators=0,
    warm_start=True,
    class_weight="balanced",
    n_jobs=-1,
    random_state=SEED,
)

t0 = time.time()
for _ in tqdm(range(N_TREES // CHUNK), desc="training RF", unit="x10 trees"):
    rf_bin.n_estimators += CHUNK
    rf_bin.fit(X_train, y_train_bin)

print(f"\ntrained {rf_bin.n_estimators} trees in {time.time()-t0:.0f}s")
joblib.dump(rf_bin, os.path.join(RESULTS, "rf_binary.joblib"))

training RF:   0%|          | 0/10 [00:00<?, ?x10 trees/s]

/usr/local/lib/python3.13/dist-packages/sklearn/ensemble/_forest.py:861: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ensemble/_forest.py:861: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the 


trained 100 trees in 874s


['/content/drive/MyDrive/AI_Cheema_Gang/processed/results/rf_binary.joblib']

In [13]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, average_precision_score

def report_binary(model, X, y, name):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    cm = confusion_matrix(y, pred)
    print(f"=== {name} ===")
    print(classification_report(y, pred, target_names=["BENIGN","ATTACK"], digits=4))
    print(f"macro F1 {f1_score(y, pred, average='macro'):.4f}   "
          f"PR-AUC {average_precision_score(y, prob):.4f}")
    print(f"missed attacks {cm[1,0]:,}   false alarms {cm[0,1]:,}\n")
    return pred

_        = report_binary(rf_bin, X_val,  y_val_bin,  "VALIDATION")
pred_bin = report_binary(rf_bin, X_test, y_test_bin, "TEST")

=== VALIDATION ===
              precision    recall  f1-score   support

      BENIGN     0.9995    0.9995    0.9995    237876
      ATTACK     0.9980    0.9979    0.9979     57636

    accuracy                         0.9992    295512
   macro avg     0.9987    0.9987    0.9987    295512
weighted avg     0.9992    0.9992    0.9992    295512

macro F1 0.9987   PR-AUC 0.9998
missed attacks 122   false alarms 117

=== TEST ===
              precision    recall  f1-score   support

      BENIGN     0.9939    0.9829    0.9884    679647
      ATTACK     0.9324    0.9753    0.9534    164697

    accuracy                         0.9814    844344
   macro avg     0.9632    0.9791    0.9709    844344
weighted avg     0.9819    0.9814    0.9815    844344

macro F1 0.9709   PR-AUC 0.9861
missed attacks 4,074   false alarms 11,645



Out of 164,697, Missed attacks (4,074)
Out of 679,647 benign flows , False alarms (11,645)
---



5-class version, where it has to name which attack.

In [16]:
from sklearn.ensemble import RandomForestClassifier
from tqdm.auto import tqdm
import time, joblib

N_TREES, CHUNK = 100, 10
rf_multi = RandomForestClassifier(n_estimators=0, warm_start=True,
    class_weight="balanced", n_jobs=-1, random_state=SEED)

t0 = time.time()
for _ in tqdm(range(N_TREES // CHUNK), desc="training RF multi", unit="x10 trees"):
    rf_multi.n_estimators += CHUNK
    rf_multi.fit(X_train, y_train_mul)
print(f"trained {N_TREES} trees in {time.time()-t0:.0f}s")

joblib.dump(rf_multi, f"{OUT_DIR}/results/rf_multiclass.joblib")

training RF multi:   0%|          | 0/10 [00:00<?, ?x10 trees/s]

/usr/local/lib/python3.13/dist-packages/sklearn/ensemble/_forest.py:861: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ensemble/_forest.py:861: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the 

trained 100 trees in 890s


['/content/drive/MyDrive/AI_Cheema_Gang/processed/results/rf_multiclass.joblib']

In [17]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score

lm = pd.read_csv(f"{OUT_DIR}/label_mapping.csv")
names = lm.sort_values(lm.columns[1])[lm.columns[0]].tolist()

def report_multi(model, X, y, name):
    pred = model.predict(X)
    labs = np.arange(len(names))
    print(f"=== {name} ===")
    print(classification_report(y, pred, labels=labs, target_names=names,
                                digits=4, zero_division=0))
    print(f"macro F1 {f1_score(y, pred, average='macro', zero_division=0):.4f}   "
          f"weighted F1 {f1_score(y, pred, average='weighted', zero_division=0):.4f}\n")
    return pred

_          = report_multi(rf_multi, X_val,  y_val_mul,  "VALIDATION")
pred_multi = report_multi(rf_multi, X_test, y_test_mul, "TEST")

=== VALIDATION ===
                            precision    recall  f1-score   support

                    BENIGN     0.9994    0.9995    0.9995    237876
                       Bot     0.9254    0.9073    0.9163       205
                      DDoS     0.9999    0.9993    0.9996     13442
             DoS GoldenEye     0.9991    1.0000    0.9995      1080
                  DoS Hulk     0.9970    0.9975    0.9973     23369
          DoS Slowhttptest     0.9330    0.9411    0.9370       577
             DoS slowloris     0.9983    1.0000    0.9992       605
               FTP-Patator     0.9988    1.0000    0.9994       833
                Heartbleed     1.0000    1.0000    1.0000         1
              Infiltration     1.0000    0.6667    0.8000         3
                  PortScan     0.9999    1.0000    1.0000     16674
               SSH-Patator     1.0000    0.9968    0.9984       619
  Web Attack - Brute Force     0.7222    0.7405    0.7312       158
Web Attack - Sql Injection  

True -> predicted

In [18]:
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_mul, pred_multi, labels=np.arange(len(names)))

print("biggest confusions (true -> predicted, count)")
pairs = []
for i in range(len(names)):
    for j in range(len(names)):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], names[i], names[j]))
for c, t, p in sorted(pairs, reverse=True)[:15]:
    print(f"{c:>8,}   {t}  ->  {p}")

biggest confusions (true -> predicted, count)
   5,434   BENIGN  ->  DDoS
   2,925   DoS Hulk  ->  BENIGN
   1,000   BENIGN  ->  PortScan
     913   DoS Hulk  ->  PortScan
     496   Bot  ->  BENIGN
     426   DoS Slowhttptest  ->  BENIGN
     400   DoS slowloris  ->  BENIGN
     184   BENIGN  ->  DoS Hulk
     124   Web Attack - XSS  ->  Web Attack - Brute Force
     118   DoS Hulk  ->  DoS GoldenEye
     112   Web Attack - Brute Force  ->  Web Attack - XSS
     109   DoS GoldenEye  ->  BENIGN
      77   DDoS  ->  BENIGN
      70   BENIGN  ->  DoS Slowhttptest
      63   PortScan  ->  BENIGN


Cross checkeking againnst the grapgh files from previous notebook

In [19]:
import pickle, torch

with open(f"{OUT_DIR}/graphs/train_5min.pkl", "rb") as f:
    train_graphs = pickle.load(f)
with open(f"{OUT_DIR}/graphs/test_5min.pkl", "rb") as f:
    test_graphs = pickle.load(f)

print("train windows", len(train_graphs))
print("test  windows", len(test_graphs))

g = train_graphs[0]
print("\none window:")
print(g)
print("\nattrs:", [k for k in g.keys()])
print("edges in window 0:", g.edge_index.shape[1])
print("total train edges:", sum(x.edge_index.shape[1] for x in train_graphs))
print("total test  edges:", sum(x.edge_index.shape[1] for x in test_graphs))

train windows 383
test  windows 173

one window:
Data(edge_index=[2, 766], edge_attr=[766, 70], y=[766], y_bin=[766], num_nodes=676)

attrs: ['y_bin', 'edge_index', 'y', 'edge_attr', 'num_nodes']
edges in window 0: 766
total train edges: 1970107
total test  edges: 844342


In [20]:
print([c for c in df.columns if c not in feature_cols])
print("\nfeature_cols:", len(feature_cols))
print("has feature_cols:", "feature_cols" in globals())
print("\nsplit3 counts:")
print(df["split3"].value_counts())

['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Label', 'day', 'day_order', 'label_binary', 'split', 'label_multi', 'src_id', 'dst_id', 'window', 'split3']

feature_cols: 70
has feature_cols: True

split3 counts:
split3
train    1674601
test      844344
val       295512
Name: count, dtype: int64


Free old grapgh files

In [21]:
import gc
for v in ["train_graphs", "test_graphs"]:
    if v in globals():
        del globals()[v]
gc.collect()

1240

we are making the grapgh again  , now on train , val , test

In [22]:
import os, pickle, gc
import numpy as np, pandas as pd, torch
from torch_geometric.data import Data
from tqdm.auto import tqdm

os.makedirs(f"{OUT_DIR}/graphs", exist_ok=True)

def build_split_graphs(sub, desc):
    out = []
    groups = list(sub.groupby(["day_order", "window"], sort=True))
    for _, g in tqdm(groups, desc=desc, unit="win"):
        nodes = pd.Index(pd.concat([g["src_id"], g["dst_id"]]).unique())
        local = pd.Series(np.arange(len(nodes)), index=nodes)
        edge_index = torch.tensor(np.vstack([
            g["src_id"].map(local).values,
            g["dst_id"].map(local).values]), dtype=torch.long)
        out.append(Data(
            edge_index=edge_index,
            edge_attr=torch.tensor(g[feature_cols].values, dtype=torch.float),
            y=torch.tensor(g["label_multi"].values, dtype=torch.long),
            y_bin=torch.tensor(g["label_binary"].values, dtype=torch.long),
            num_nodes=len(nodes)))
    return out

summary = []
for name in ["train", "val", "test"]:
    sub = df[df["split3"] == name]
    gs = build_split_graphs(sub, f"building {name}")
    edges = sum(g.edge_index.shape[1] for g in gs)
    with open(f"{OUT_DIR}/graphs/{name}_5min_v2.pkl", "wb") as f:
        pickle.dump(gs, f, protocol=4)
    summary.append((name, len(gs), edges, len(sub), edges == len(sub)))
    del gs
    gc.collect()

print()
for name, nwin, edges, expected, ok in summary:
    print(f"{name:<6} windows {nwin:>4}   edges {edges:>9,}   expected {expected:>9,}   match {ok}")

building train:   0%|          | 0/326 [00:00<?, ?win/s]

building val:   0%|          | 0/89 [00:00<?, ?win/s]

building test:   0%|          | 0/175 [00:00<?, ?win/s]


train  windows  326   edges 1,674,601   expected 1,674,601   match True
val    windows   89   edges   295,512   expected   295,512   match True
test   windows  175   edges   844,344   expected   844,344   match True


In [23]:
import torch
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU: False
